In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import dataclasses
import json

import numpy as onp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from msmjax.calculators import StaticCellMSMParams, static_cell_msm
from msmjax.convenience import suggest_msm_params
from msmjax.benchmark_tools import (
    path_input_structures,
    evaluate_structure_with_lammps_p3m,
)

In [2]:
PBC = (False, False, False)
LEVEL_ONE_GRIDSPACING = 1.0
LEVEL_ZERO_CUTOFF = 3.0

In [3]:
structures = onp.load(path_input_structures / "structures_10000.npz")

In [4]:
(n_particles, n_dim) = structures["positions"][0].shape
cell = structures["cells"][0]
pos = structures["positions"][0]
chg = structures["charges"][0]
cell = cell.astype(onp.float64)
pos = pos.astype(onp.float64)
chg = chg.astype(onp.float64)

In [5]:
suggested_params = suggest_msm_params(
    box_lengths=onp.diag(cell),
    pbc=PBC,
    n_particles=n_particles,
    level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
    level_zero_cutoff=LEVEL_ZERO_CUTOFF,
)
suggested_params

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")



{'level_one_gridspacing': [1.0, 1.0, 1.0],
 'level_zero_cutoff': 3.0,
 'p': 4,
 'mu': 14,
 'n_levels': 5}

In [6]:
max_level_split = suggested_params["n_levels"]
if not onp.any(PBC):
    max_level_eval = max_level_split
elif onp.all(PBC):
    max_level_eval = max_level_split - 1
else:
    raise ValueError("Mixed BCs currently not handled.")

params_dataclass = StaticCellMSMParams(
    p=suggested_params["p"],
    mu=suggested_params["mu"],
    r_cut_0=suggested_params["level_zero_cutoff"],
    h_1=suggested_params["level_one_gridspacing"],
    max_level_split=max_level_split,
    max_level_eval=max_level_eval,
    cell=cell,
    cell_mode="ortho",
    pbc=PBC,
    supercell_diag=None,
    use_neighborlist=False,
    convolution_methods="scipy-fft",  # TODO
    grids_defined_on_unitcube=False,  # TODO
)

In [7]:
[f.name for f in dataclasses.fields(params_dataclass)]

['p',
 'mu',
 'r_cut_0',
 'h_1',
 'max_level_split',
 'max_level_eval',
 'cell',
 'cell_mode',
 'pbc',
 'supercell_diag',
 'use_neighborlist',
 'convolution_methods',
 'grids_defined_on_unitcube']

In [8]:
(
    calc_energy,
    calc_forces,
    calc_energy_and_forces,
    calc_charge_gradient,
) = static_cell_msm(params_dataclass)

calc_energy = jax.jit(calc_energy)
calc_forces = jax.jit(calc_forces)

In [9]:
# calc_energy(pos, chg, neighborlist=jnp.array([1., 2., 3.]))
calc_energy(pos, chg)

Array(-40.38278237, dtype=float64)

In [10]:
# calc_forces(pos, chg, neighborlist=jnp.array([1.0, 2.0, 3.0]))
calc_forces(pos, chg)

Array([[ 0.73853043, -0.24289055, -0.35225429],
       [-1.23556034, -0.74068227, -0.23327418],
       [-1.12392592, -2.55162579, -0.38015646],
       ...,
       [-0.30609247, -0.78932256, -0.24524063],
       [ 0.91776981,  1.79724765, -0.05913628],
       [ 1.27772065, -0.64229829, -1.76050789]], dtype=float64)

In [11]:
with open("params.json", "w") as f:
    json.dump(params_dataclass, f)

TypeError: Object of type StaticCellMSMParams is not JSON serializable